In [10]:
from sklearn.cluster import KMeans
from scipy.stats import chi2_contingency
import polars.selectors as cs
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
import polars as pl

In [19]:
df_cleaned = pl.read_parquet("data/cleaned.parquet")
df_features = pl.read_parquet("data/rule_classified.parquet")

In [20]:
df_features.head()

source.ip,3xx,2xx,5xx,4xx,active_days_ratio,request_count,endpoint_entropy,time_entropy,behavior_label
str,f64,f64,f64,f64,f64,u32,f64,f64,str
"""172.105.89.161""",1.0,0.0,0.0,0.0,0.714286,6,-0.0,2.584963,"""other"""
"""186.32.219.176""",1.0,0.0,0.0,0.0,0.142857,1,-0.0,-0.0,"""background_noise"""
"""1.180.246.82""",1.0,0.0,0.0,0.0,0.142857,1,-0.0,-0.0,"""background_noise"""
"""185.191.171.44""",0.0,1.0,0.0,0.0,0.142857,1,-0.0,-0.0,"""background_noise"""
"""185.191.171.40""",0.0,1.0,0.0,0.0,0.142857,1,-0.0,-0.0,"""background_noise"""


## Feature extraction for model

time from last request - nulls imputed with maximum value in the dataset

In [12]:
df_sorted = df_cleaned.sort(["source.ip", "timestamp"])

# 2. Calculate the time difference within each source.ip group
df_with_diff = df_sorted.with_columns(
    pl.col("timestamp")
    .diff()
    .over("source.ip")
    .alias("time.last_request") 
)

# 3. Calculate the max duration in a single expression
max_duration = df_with_diff.get_column("time.last_request").drop_nulls().max()
print(f"Maximum observed time difference: {max_duration}")

# 4. Impute nulls with the max duration AND convert to seconds in one step
df_with_diff = df_with_diff.with_columns(
    pl.col("time.last_request")
    .fill_null(max_duration)
    .dt.total_seconds()
    .alias("time.last_request.seconds")
)

Maximum observed time difference: 5 days, 20:22:47


In [13]:
df_with_diff.head(3)

timestamp,source.ip,source.registered_domain,source.as,source.geo,destination.ip,http.request.method,http.response.status_code,http.response.bytes,url.path,url.filetype,user_agent.name,user_agent.version,user_agent.os.platform,user_agent.os.device.name,user_agent.category,agent.name,agent.id,tags,correlation,geo_code,time.last_request,time.last_request.seconds
datetime[ms],str,str,struct[2],struct[4],str,str,str,i64,str,str,str,str,str,str,str,str,str,list[str],struct[1],str,duration[ms],i64
2020-12-25 13:18:51,"""1.10.131.122""",null,"{""TOT Public Company Limited"",23969}","{{13.8583,100.4688},""Phra Samut Chedi"",""Thailand"",""TH""}","""116.203.157.62""","""GET""","""301""",162,"""/""","""""","""Safari""","""9""","""Mac OS X""","""Other""","""BROWSER""","""gitlab""","""ad99edda-a068-4794-b2c5-201664…","[""beats_input_codec_plain_applied"", ""file_backup""]",{[]},"""TH""",null,505367
2020-12-27 03:25:57,"""1.180.246.82""",null,"{""Chinanet"",4134}","{{39.1423,117.1726},""Tianjin"",""China"",""CN""}","""116.203.157.62""","""GET""","""301""",162,"""/manager/html/""","""""",null,null,null,null,null,"""gitlab""","""ad99edda-a068-4794-b2c5-201664…","[""beats_input_codec_plain_applied"", ""file_backup""]","{[""AQVT52HL""]}","""CN""",null,505367
2020-12-26 22:07:36,"""1.48.51.149""",null,"{""Chinanet"",4134}","{{34.7725,113.7266},null,""China"",""CN""}",null,"""GET""","""301""",169,"""/""","""""",null,null,null,null,null,null,"""bf7ed7ab-e33b-4bd2-adb1-cb5ff9…","[""beats_input_codec_plain_applied"", ""_geoip_lookup_failure"", ""file_backup""]",{[]},"""CN""",null,505367


complexity of URL

In [ ]:
# Assuming 'df_cleaned' is your Polars DataFrame
df_different_urls = df_cleaned.filter(
    pl.col("url.path") != pl.col("url.original")
)

# Display the rows where the path and original URL differ
print(df_different_urls.select(["url.path", "url.original"]))

shape: (4_981, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ url.path                        ┆ url.original                    │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ /oauth/authorize                ┆ /oauth/authorize?acr_values=&a… │
│ /securely/common/securely-app-… ┆ /securely/common/securely-app-… │
│ /index.php                      ┆ /index.php?s=/Index/\x5Cthink\… │
│ /                               ┆ /?XDEBUG_SESSION_START=phpstor… │
│ /index.php                      ┆ /index.php?s=/Index/\x5Cthink\… │
│ /groups/securely/-/milestones   ┆ /groups/securely/-/milestones?… │
│ /securely/common/secrule-confi… ┆ /securely/common/secrule-confi… │
│ /securely/common/securely-app-… ┆ /securely/common/securely-app-… │
│ /securely/front-end/securely-o… ┆ /securely/front-end/securely-o… │
│ 

In [ ]:
# Assuming 'df_cleaned' is your Polars DataFrame
df_with_diff = df_with_diff.with_columns(
    pl.col("url.path")
    .str.count_matches('/') # Count occurrences of '/'
    .alias("url.deepnes")   # Name the new column
)

# Display the result
print(df_with_diff.select(["url.path", "url.deepnes"]).head())

shape: (5, 2)
┌────────────────┬─────────────┐
│ url.path       ┆ url.deepnes │
│ ---            ┆ ---         │
│ str            ┆ u32         │
╞════════════════╪═════════════╡
│ /              ┆ 1           │
│ /manager/html/ ┆ 3           │
│ /              ┆ 1           │
│ /              ┆ 1           │
│ /              ┆ 1           │
└────────────────┴─────────────┘


In [ ]:
# if sql injection in url or complex patterns
df_with_diff = df_with_diff.with_columns(
    pl.col("url.original")
    .str.extract(r"\?(.*)", 1) 
    .str.len_chars()         
    .fill_null(0)             
    .alias("url.query.length")
)

In [ ]:
# how many query parameters in url
df_with_diff = df_with_diff.with_columns(
    pl.when(pl.col("url.original").str.contains(r"\?", literal=False))
    .then(
        pl.col("url.original")
        .str.extract(r"\?(.*)", 1) # Get query string
        .str.count_matches('&')    # Count '&'
        + 1                        # Add 1 for the first parameter
    )
    .otherwise(0)                  # 0 parameters if no '?'
    .alias("url.number.params")
)

distance from rolling mean

In [ ]:
def join_on_bins(df1, df2, column):
    df1 = df1.lazy().with_columns(
        pl.col("timestamp").dt.date().cast(pl.String).alias("day"), # Cast to String
        pl.col("timestamp").dt.truncate("5m").dt.time().alias("time_bin")
    )

    df2 = df2.lazy().select(
        pl.col("day").dt.date().cast(pl.String).alias("day"), # Cast to String 
        pl.col("time_bin"),
        pl.col(column)
    )

    df1 = df1.join(
        df2,
        on=["day", "time_bin"],
        how="left"
    ).collect() # Execute the join

    df1 = df1.drop(["day", "time_bin"])
    return df1

In [ ]:
#Calculate the distances (absolute and percentage)
df_with_distance = df_with_rolling.with_columns(
    # Absolute difference
    (pl.col("len") - pl.col("baseline")).abs().alias("rolling.mean.distance"),

    # Percentage difference (handle baseline=0 to avoid errors)
    pl.when(pl.col("baseline") > 0)
    .then( ((pl.col("len") - pl.col("baseline")) / pl.col("baseline")) * 100 )
    .otherwise(None) # Or pl.lit(0) if you prefer 0 for zero baselines
    .alias("rolling.mean.distance.pct")
)


In [ ]:
df_with_diff = join_on_bins(df_with_diff, df_with_distance, "rolling.mean.distance.pct")

In [ ]:
# Add the join keys ('day', 'time_bin') to df_cleaned (lazily)
df_with_diff = df_with_diff.lazy().with_columns(
    pl.col("timestamp").dt.date().cast(pl.String).alias("day"), # Cast to String
    pl.col("timestamp").dt.truncate("5m").dt.time().alias("time_bin")
)

In [ ]:
# Prepare distance_to_join, ensuring 'day' is also String
distance_to_join = df_with_distance.lazy().select(
    pl.col("day").dt.date().cast(pl.String).alias("day"), # Keep as String (it's already String in df_with_distance)
    pl.col("time_bin"),
    pl.col("rolling.mean.distance"),
    pl.col("rolling.mean.distance.pct")
)

# Perform the left join (String == String should work now)
df_with_diff = df_with_diff.join(
    distance_to_join,
    on=["day", "time_bin"],
    how="left"
).collect() # Execute the join

# (Optional) Drop the temporary key columns if not needed
df_with_diff = df_with_diff.drop(["day", "time_bin"])

In [ ]:
df_with_diff.get_column("user_agent.os.device.name").value_counts(sort=True).head(5)

user_agent.os.device.name,count
str,u32
"""Other""",683724
"""Spider""",5472
"""Generic Smartphone""",1994
null,226
"""iPhone""",86


In [ ]:
# One-hot encode user_agent.category, handling nulls
df_with_diff = df_with_diff.with_columns(
    pl.col("user_agent.category").fill_null("UNKNOWN")
)
df_with_diff = df_with_diff.to_dummies(columns=["user_agent.category"])

rolling distribution of requst types

In [ ]:
df_binned = df_cleaned.with_columns(
    pl.col("timestamp").dt.truncate("5m").alias("time_bin")
)

df_binned = (
    df_binned.group_by("time_bin")
    .agg(
        pl.col("http.request.method").value_counts()
    )
    .explode("http.request.method") # Explode the list of structs
    .unnest("http.request.method")   # Unnest the struct columns
    .pivot(             # Pivot to get methods as columns
        index="time_bin",
        on="http.request.method",
        values="count"
    )
    .fill_null(0) # Fill missing methods in a bin with 0 count
    .sort("time_bin")
)

method_cols = [col for col in df_binned.columns if col != "time_bin"]
df_binned = df_binned.with_columns(
    pl.sum_horizontal(method_cols).alias("total_requests")
)

# Divide each method count by the total to get proportion
# (Handle division by zero if a bin has 0 requests)
df_binned = df_binned.with_columns(
    [
        (pl.when(pl.col("total_requests") > 0)
         .then(pl.col(method) / pl.col("total_requests"))
         .otherwise(0) # Proportion is 0 if total is 0
         .alias(f"{method}_prop")
        ) for method in method_cols
    ]
).select(["time_bin"] + [f"{method}_prop" for method in method_cols]) # Keep only proportion columns

print("--- Proportions per 5-min Bin ---")
print(df_binned.head(3))

--- Proportions per 5-min Bin ---
shape: (3, 8)
┌─────────────┬──────────┬───────────┬───────────┬────────────┬──────────┬────────────┬────────────┐
│ time_bin    ┆ GET_prop ┆ HEAD_prop ┆ POST_prop ┆ PATCH_prop ┆ PUT_prop ┆ OPTIONS_pr ┆ DELETE_pro │
│ ---         ┆ ---      ┆ ---       ┆ ---       ┆ ---        ┆ ---      ┆ op         ┆ p          │
│ datetime[μs ┆ f64      ┆ f64       ┆ f64       ┆ f64        ┆ f64      ┆ ---        ┆ ---        │
│ , UTC]      ┆          ┆           ┆           ┆            ┆          ┆ f64        ┆ f64        │
╞═════════════╪══════════╪═══════════╪═══════════╪════════════╪══════════╪════════════╪════════════╡
│ 2020-12-21  ┆ 0.711864 ┆ 0.00565   ┆ 0.282486  ┆ 0.0        ┆ 0.0      ┆ 0.0        ┆ 0.0        │
│ 00:00:00    ┆          ┆           ┆           ┆            ┆          ┆            ┆            │
│ UTC         ┆          ┆           ┆           ┆            ┆          ┆            ┆            │
│ 2020-12-21  ┆ 0.711864 ┆ 0.00565   ┆ 0.28

In [ ]:
# Define the rolling window size (e.g., 12 bins = 1 hour)
window_size = 12
prop_cols = [col for col in df_binned.columns if col.endswith("_prop")]

df_rolling = df_binned.with_columns(
    [
        pl.col(prop_col).rolling_mean(
            window_size=window_size,
            min_samples=1 # Start calculating even if window isn't full
        ).alias(f"{prop_col}_rolling_avg")
        for prop_col in prop_cols
    ]
)

print("\n--- Rolling Average Proportions ---")
print(df_rolling.tail(3))


--- Rolling Average Proportions ---
shape: (3, 15)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┬───────┬───────┬───────┐
│ tim ┆ GET ┆ HEA ┆ POS ┆ PAT ┆ PUT ┆ OPT ┆ DEL ┆ GET ┆ HEA ┆ POS ┆ PATCH_ ┆ PUT_p ┆ OPTIO ┆ DELET │
│ e_b ┆ _pr ┆ D_p ┆ T_p ┆ CH_ ┆ _pr ┆ ION ┆ ETE ┆ _pr ┆ D_p ┆ T_p ┆ prop_r ┆ rop_r ┆ NS_pr ┆ E_pro │
│ in  ┆ op  ┆ rop ┆ rop ┆ pro ┆ op  ┆ S_p ┆ _pr ┆ op_ ┆ rop ┆ rop ┆ olling ┆ ollin ┆ op_ro ┆ p_rol │
│ --- ┆ --- ┆ --- ┆ --- ┆ p   ┆ --- ┆ rop ┆ op  ┆ rol ┆ _ro ┆ _ro ┆ _avg   ┆ g_avg ┆ lling ┆ ling_ │
│ dat ┆ f64 ┆ f64 ┆ f64 ┆ --- ┆ f64 ┆ --- ┆ --- ┆ lin ┆ lli ┆ lli ┆ ---    ┆ ---   ┆ _avg  ┆ avg   │
│ eti ┆     ┆     ┆     ┆ f64 ┆     ┆ f64 ┆ f64 ┆ g_a ┆ ng_ ┆ ng_ ┆ f64    ┆ f64   ┆ ---   ┆ ---   │
│ me[ ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ vg  ┆ avg ┆ avg ┆        ┆       ┆ f64   ┆ f64   │
│ μs, ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ --- ┆ --- ┆ --- ┆        ┆       ┆       ┆       │
│ UTC ┆     ┆     ┆     ┆     ┆     ┆  

In [ ]:
# Calculate Manhattan Distance
# List of difference expressions for each method type
diff_exprs = [
    (pl.col(prop_col) - pl.col(f"{prop_col}_rolling_avg")).abs()
    for prop_col in prop_cols
]

# Sum the absolute differences horizontally
df_final_feature = df_rolling.with_columns(
    pl.sum_horizontal(diff_exprs).alias("method_dist_change")
).select(["time_bin", "method_dist_change"]) # Keep only the final feature

print("\n--- Final Distribution Change Feature ---")
print(df_final_feature.head())


--- Final Distribution Change Feature ---
shape: (5, 2)
┌─────────────────────────┬────────────────────┐
│ time_bin                ┆ method_dist_change │
│ ---                     ┆ ---                │
│ datetime[μs, UTC]       ┆ f64                │
╞═════════════════════════╪════════════════════╡
│ 2020-12-21 00:00:00 UTC ┆ 0.0                │
│ 2020-12-21 00:05:00 UTC ┆ 0.0                │
│ 2020-12-21 00:10:00 UTC ┆ 0.002183           │
│ 2020-12-21 00:15:00 UTC ┆ 0.000406           │
│ 2020-12-21 00:20:00 UTC ┆ 0.000981           │
└─────────────────────────┴────────────────────┘


In [ ]:
df_final_feature_formatted = df_final_feature.with_columns(
    pl.col("time_bin").dt.date().alias("day"),
    pl.col("time_bin").dt.time().alias("time_bin") 
).select(
    ["time_bin", "day", "method_dist_change"]
)

In [ ]:
df_with_diff = join_on_bins(df_with_diff, df_final_feature_formatted, "method_dist_change")

In [ ]:
# numeric_df = df_with_diff.select(cs.numeric())
# numeric_df.describe()

In [ ]:
# "http.response.bytes" needs log transform cause too big max values
df_with_diff = df_with_diff.with_columns(
    (pl.col("http.response.bytes") + 1).log().alias("http.response.bytes.log")
).drop("http.response.bytes")

there was not enough time but I wanted to extract more features:
- location based request (distance between source and host)
- estimate "sessions" by grouping requests from the same source.ip that are close in time within 30 minutes. How many requests are in a typical session? Anomalies might have very long or very short sessions.

# ML models

In [ ]:
NUMERIC_FEATURES = [
    "time.last_request.seconds",
    "url.deepnes",
    "url.query.length",
    "rolling.mean.distance.pct",
    "method_dist_change",
    "http.response.bytes.log",
    "url.number.params"
]
BINARY_FEATURES = [
    "user_agent.category_SCRIPTING",
    "user_agent.category_BROWSER",
    "user_agent.category_UNKNOWN" # Assuming nulls were mapped to UNKNOWN
]

MODEL_FEATURES = NUMERIC_FEATURES + BINARY_FEATURES

In [ ]:
# Create the feature matrix
X_df = df_with_diff.select(MODEL_FEATURES)

# Final check for nulls and imputation for the model features
for col_name in X_df.columns:
    if X_df[col_name].null_count() > 0:
        median_val = X_df[col_name].median()
        X_df = X_df.with_columns(
            pl.col(col_name).fill_null(median_val)
        )

X = X_df.to_numpy()

In [ ]:
# Separate numerical and binary parts
X_numeric = X[:, :len(NUMERIC_FEATURES)]
X_binary = X[:, len(NUMERIC_FEATURES):]

# Scale the numerical features
scaler = StandardScaler()
X_scaled_numeric = scaler.fit_transform(X_numeric)

# Combine the scaled numeric features with the unscaled binary features
X_final = np.hstack([X_scaled_numeric, X_binary])

print(f"Final feature matrix shape: {X_final.shape}")

Final feature matrix shape: (691505, 10)


# Isolation Forest - simplest

In [ ]:
# Initialize Isolation Forest
# n_estimators: number of trees to build
# contamination: estimated fraction of outliers in the dataset
iso_forest = IsolationForest(
    n_estimators=100, 
    contamination=0.005, # Assuming 0.5% of traffic is malicious/anomalous
    random_state=42
)

print("Training Isolation Forest...")
iso_forest.fit(X_final)
print("Training complete.")

Training Isolation Forest...
Training complete.


In [ ]:
# Generate raw anomaly scores
raw_scores = iso_forest.decision_function(X_final)

# Invert and normalize the score for interpretability (higher score = more anomalous)
anomaly_scores = -raw_scores

# Add Score to dataframe
df_anomalies = df_with_diff.with_columns(
    pl.Series("anomaly_score", anomaly_scores)
)

print("\n--- Top 5 Most Anomalous Log Records (Highest Scores) ---")
print(df_anomalies.sort("anomaly_score", descending=True).select([
    "timestamp", 
    "source.ip", 
    "anomaly_score", 
    "rolling.mean.distance.pct"
]).head(5))


--- Top 5 Most Anomalous Log Records (Highest Scores) ---
shape: (5, 4)
┌─────────────────────────┬────────────────┬───────────────┬───────────────────────────┐
│ timestamp               ┆ source.ip      ┆ anomaly_score ┆ rolling.mean.distance.pct │
│ ---                     ┆ ---            ┆ ---           ┆ ---                       │
│ datetime[μs, UTC]       ┆ str            ┆ f64           ┆ f64                       │
╞═════════════════════════╪════════════════╪═══════════════╪═══════════════════════════╡
│ 2020-12-24 17:35:19 UTC ┆ 18.205.117.84  ┆ 0.072386      ┆ 20.673813                 │
│ 2020-12-22 11:15:59 UTC ┆ 188.103.82.202 ┆ 0.072386      ┆ 47.887324                 │
│ 2020-12-22 15:13:15 UTC ┆ 66.249.66.63   ┆ 0.070715      ┆ 48.931384                 │
│ 2020-12-21 11:32:21 UTC ┆ 114.119.150.67 ┆ 0.066276      ┆ 37.891738                 │
│ 2020-12-24 14:54:17 UTC ┆ 161.35.89.202  ┆ 0.065722      ┆ 13.432836                 │
└─────────────────────────┴──────────

In [ ]:
threshold_score = np.percentile(anomaly_scores, 99.5) # The 99.5th percentile score

# Flag potential attacks
df_anomalies = df_anomalies.with_columns(
    (pl.col("anomaly_score") > threshold_score).alias("attack")
)

print(f"\nAnomaly Score Threshold (P99.5): {threshold_score:.4f}")
print(f"Total potential attacks flagged: {df_anomalies['attack'].sum()}")


Anomaly Score Threshold (P99.5): 0.0000
Total potential attacks flagged: 3453


Isolation Forest Anomaly Scores
Here's how the raw scores from the Isolation Forest model are interpreted:

- Scores close to 1 are considered Normal (not anomalous).
- Scores close to -1 are considered Anomalous (outliers).
- Scores close to 0 mean the point is on the boundary (not clearly inside or outside the distribution).

The model finetuning is the next step. Then analysis of results in order to explain potential attacking logs.

# Local Outlier Factor (LOF) model

In [ ]:
# 1. Create a binary column for 4xx errors
df_with_4xx_flag = df_with_diff.with_columns(
    (pl.col("http.response.status_code").cast(pl.Int64).floordiv(100) == 4)
    .cast(pl.UInt8)
    .alias("is_4xx_error")
)

# 2. Calculate the mean (rate) of 4xx errors for each source IP
ip_error_rates = df_with_4xx_flag.group_by("source.ip").agg(
    pl.col("is_4xx_error").mean().alias("rate_4xx")
)

# 3. Join the rate back to the main DataFrame
df_lof_features = df_with_4xx_flag.join(
    ip_error_rates,
    on="source.ip",
    how="left"
)

print("Added 'rate_4xx' feature.")

Added 'rate_4xx' feature.


In [ ]:
df_lof_features.columns

['http.request.method',
 'agent.name',
 'destination.geo',
 'user_agent.name',
 'tags',
 'source.as',
 'service.type',
 'correlation',
 'url.controller',
 'user_agent.description',
 'agent.hostname',
 'user_agent.original',
 'agent.version',
 'user_agent.os.platform',
 'event.created',
 'source.domain',
 'user_agent.category_BROWSER',
 'user_agent.category_SCRIPTING',
 'user_agent.category_UNKNOWN',
 'user_agent.version',
 'agent.ephemeral_id',
 'source.registered_domain',
 'user_agent.os.device.name',
 'url.original',
 'service.name',
 'url.path',
 'ecs.version',
 'url.filetype',
 'source.ip',
 'http.response.status_code',
 'http.version',
 'destination.ip',
 'http.response.status',
 'source.geo',
 'agent.id',
 'destination.as',
 'source.network',
 'timestamp',
 'time.last_request',
 'time.last_request.seconds',
 'url.deepnes',
 'url.query.length',
 'url.number.params',
 'rolling.mean.distance.pct',
 'rolling.mean.distance',
 'rolling.mean.distance.pct_right',
 'method_dist_change',
 

In [ ]:
# Define the features required by the LOF model plan
LOF_FEATURES = [
        "time.last_request.seconds",
        "rate_4xx",
        "http.response.bytes.log" # This assumes the log-transformed column name is now 'log_response_bytes'
    ]
X_lof_df = df_lof_features.select(LOF_FEATURES)

scaler_lof = StandardScaler()
X_lof_scaled = scaler_lof.fit_transform(X_lof_df)

In [ ]:
# Initialize LOF model
# n_neighbors: How many neighbors to consider. 20 is a good starting point.
# contamination: Used only for prediction labeling, not model fitting.
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination='auto', # 'auto' is the default for fitting when no ground truth is available
    novelty=False # Set to False for anomaly detection (not novelty detection)
)

# Train the model and get the raw anomaly scores (Negative Outlier Factor)
print("Training LOF Model...")
negative_outlier_factor = lof.fit_predict(X_lof_scaled) # Returns -1 for inliers, 1 for outliers based on internal threshold

# Get the raw negative outlier factor scores (higher magnitude means more local deviation)
raw_scores_lof = lof.negative_outlier_factor_

# Invert scores for intuitive ranking (Higher score = More Anomalous)
anomaly_scores_lof = -raw_scores_lof

# 4. Add Score to DataFrame
df_anomalies = df_lof_features.with_columns(
    pl.Series("lof_anomaly_score", anomaly_scores_lof)
)

# 5. Review Top Anomalies
print("\n--- Top 5 Most Anomalous Log Records (Highest LOF Scores) ---")
print(df_anomalies.sort("lof_anomaly_score", descending=True).select([
    "timestamp",
    "source.ip",
    "rate_4xx",
    "lof_anomaly_score"
]).head(5))

Training LOF Model...

--- Top 5 Most Anomalous Log Records (Highest LOF Scores) ---
shape: (5, 4)
┌─────────────────────────┬────────────────┬──────────┬───────────────────┐
│ timestamp               ┆ source.ip      ┆ rate_4xx ┆ lof_anomaly_score │
│ ---                     ┆ ---            ┆ ---      ┆ ---               │
│ datetime[μs, UTC]       ┆ str            ┆ f64      ┆ f64               │
╞═════════════════════════╪════════════════╪══════════╪═══════════════════╡
│ 2020-12-21 11:27:17 UTC ┆ 212.61.100.37  ┆ 0.012016 ┆ 8.0241e10         │
│ 2020-12-22 14:03:39 UTC ┆ 89.205.138.8   ┆ 0.00674  ┆ 5.1985e10         │
│ 2020-12-21 08:05:55 UTC ┆ 86.83.227.6    ┆ 0.017341 ┆ 4.4699e10         │
│ 2020-12-26 19:29:25 UTC ┆ 65.60.11.210   ┆ 0.0      ┆ 3.5054e10         │
│ 2020-12-25 23:33:22 UTC ┆ 167.179.103.33 ┆ 0.0      ┆ 3.1584e10         │
└─────────────────────────┴────────────────┴──────────┴───────────────────┘


d:\projekty\securely\.venv\Lib\site-packages\sklearn\neighbors\_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


The next step is to finetune the model, try to explain the results and comparison between the models